# A2 worker (Kaggle)
Runs this person's share of the A2 experiments (SegFormer-B0 vs DeepLabV3+ MobileNetV2 on ADE20K). All logic is in `A2/a2lib`; protocol: `A2/README.md`.

**Settings** (right side): Accelerator **GPU T4 x2**, Internet **On**, *Add-ons → Secrets* → `HF_TOKEN` (write token for `dl4dl/a1-store`). **No dataset input is needed**: ADE20K is downloaded inside the session (its terms forbid redistribution) and checked against its SHA-256.

**Order:** (1) set `RUN_SMOKE = True` once per account and run cells 1-4. (2) The sweep runs (`a2_sw_*`) come first; when both members finished them, ONE person runs `python -m a2lib.plan choose-lr`, commits `A2/results/lr_choice_a2.json`. (3) The main runs unlock automatically. Stopping and restarting is always safe.

In [ ]:
# 1) your settings
OWNER = "dat"                         # "dat" or "ly"
GPUS = 2
MAX_HOURS = 8.5                       # clean stop (checkpoint pushed) before Kaggle's 12 h limit
HF_REPO = "dl4dl/a1-store"
REPO_URL = "https://github.com/broistg/deep-learning-CO5085.git"
BRANCH = "main"
STEAL = False
RUN_SMOKE = True                      # first time on an account: downloads ADE20K (~1 GB), trains 2 x 4 steps, writes nothing to the store


In [ ]:
# 2) set up (torch/torchvision are preinstalled and are not touched)
import os, subprocess, sys
REPO = "/kaggle/working/repo"
def sh(c): print("$", c); subprocess.run(c, shell=True, check=True)
if not os.path.isdir(REPO + "/.git"):
    sh(f"git clone -b {BRANCH} {REPO_URL} {REPO}")
else:
    sh(f"cd {REPO} && git pull --ff-only")
os.chdir(REPO)
sh("pip install -q -r requirements/a2.txt && pip install -q -e .")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ.update(A2_STORE=f"hf:{HF_REPO}", A1_STORE=f"hf:{HF_REPO}", A1_OWNER=OWNER, A2_LOCAL="/kaggle/temp/a2_local",
                  PYTHONPATH=f"{REPO}:{REPO}/A1:{REPO}/A2", A1_PUSH_EVERY_MIN="45")
sh("git log -1 --format='commit %h %s'; nvidia-smi -L; df -h /kaggle/temp | tail -1")
sh("PYTHONPATH=.:A1:A2 python -m unittest discover -s A2/tests")


In [ ]:
# 3) status (read-only)
sh("python -m a2lib.worker status --a2-dir A2 | tail -30")


In [ ]:
# 4) smoke test: one SegFormer and one DeepLab run, pretrained, 2 epochs x 4 steps, real data. Prints losses, dev/test mIoU, speed.
if RUN_SMOKE:
    sh("python -m a2lib.worker prepare")
    for cfg in ("a2_m_sf_pretrained_s0", "a2_m_dl_pretrained_s0"):
        sh(f"python -m a2lib.train --config A2/configs/{cfg}.yaml --smoke --owner {OWNER}")


In [ ]:
# 5) THE LONG CELL
steal = "--steal" if STEAL else ""
sh(f"python -m a2lib.worker launch --owner {OWNER} --a2-dir A2 --gpus {GPUS} --max-hours {MAX_HOURS} {steal}")


In [ ]:
# 6) after the session
sh("python -m a2lib.worker status --a2-dir A2 | tail -30")
